### Explanation of Each Import

- **`os`**: Helps Python work with the operating system, such as file paths and folders.

- **`certifi`**: Provides trusted certificates(SSL) that help Python check if a server is real when making a secure internet connection.

- **`requests`**: Sends requests to websites and APIs to get or send data over the internet.

- **`load_dotenv`**: Loads environment variables from a `.env` file, such as API keys, so they can be used in Python.

- **`ChatGoogleGenerativeAI`**: Allows our Python code to communicate with Google's AI models, such as Gemini.

- **`tool`**: Allows us to turn a Python function into a tool that an AI agent can use to perform a task.

- **`TavilySearchResults`**: Allows the agent to search the internet using Tavily and get search results. This helps the agent find information that may not be available in the model's existing knowledge.

- **`create_react_agent`**: Creates an agent that follows the ReAct(React-Action) approach. The agent can think about the task, choose a tool, observe the result, and decide what to do next.

- **`AgentExecutor`**: Runs the agent and manages its interaction with tools until it finishes the task or reaches a stopping limit. Think of it as the component that runs the agent's process.

- **`hub`**: Connects to LangChain Hub, where we can find and download reusable prompts. A prompt tells the AI how to behave and what task to perform.

### How They Work Together

1. `load_dotenv` loads API keys.
2. `ChatGoogleGenerativeAI` connects the code to Gemini.
3. `TavilySearchResults` provides an internet search tool.
4. `tool` allows us to create our own tools.
5. `create_react_agent` combines the AI model, tools, and prompt to create an agent.
6. `AgentExecutor` runs the agent and manages its actions.
7. `os`, `certifi`, and `requests` support file handling and internet communication where needed.
8. `hub` helps us get a reusable prompt.



In [1]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

## 2. **Load Environment Variables and Configure Certificates**

- `SSL_CERT_FILE`: Stores the path to the certificate file used for secure connections.
- `load_dotenv()`: Loads environment variables from the `.env` file.
- `os.getenv()`: Retrieves environment variables, such as API keys.
- `GOOGLE_API_KEY`: Used to access Google's Gemini models.
- `TAVILY_API_KEY`: Used to access Tavily's search service.
- `WEATHERSTACK_API_KEY`: Used to access Weatherstack's weather data service.

In [2]:
# Set the certificate file path and load environment variables
os.environ['SSL_CERT_FILE'] = certifi.where()
load_dotenv()

gemini_api_key = os.getenv("GOOGLE_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")
weatherstack_api_key = os.getenv("WEATHERSTACK_API_KEY")

## 3. **Define the Tavily Search Tool**

Tavily searches the internet and returns relevant results.

- `TavilySearchResults`: Creates a tool for internet searches.
- `max_results=5`: Sets the maximum number of search results to return per search.


In [3]:
tavily_tool = TavilySearchResults(max_results=5)

## 4. Define the Weather Tool

- `@tool`: Converts the Python function into a tool that an AI agent can use.
- `get_weather_data(city: str)`: Defines a function that accepts a city name as a string.
- `-> str`: Specifies that the function returns a string.
- `url`: Stores the Weatherstack API URL, including the API key and city name.
- `requests.get(url)`: Sends a request to the Weatherstack API.
- `response.json()`: Converts the API response into a Python dictionary.
- `if 'current' not in data`: Checks whether current weather data is available.
- `return`: Sends the weather information or an error message back to the caller.
- `Temperature`: Displays the temperature in Celsius (`°C`).
- `Weather`: Displays the weather description.
- `Humidity`: Displays the humidity percentage.


**NOTE:** The Weatherstack API returns a lot of information. To understand the complete response, you can temporarily comment out the `return` statements in your function and print the `data` variable. Alternatively, you can look at the example below.

**Example API Response:**

```json
{
    "request": {
        "type": "City",
        "query": "New York, United States of America",
        "language": "en",
        "unit": "m"
    },
    "location": {
        "name": "New York",
        "country": "United States of America",
        "region": "New York",
        "lat": "40.714",
        "lon": "-74.006",
        "timezone_id": "America/New_York",
        "localtime": "2026-10-14 17:42",
        "localtime_epoch": 1791999720,
        "utc_offset": "-4.0"
    },
    "current": {
        "observation_time": "09:42 PM",
        "temperature": 26,
        "weather_code": 122,
        "weather_icons": [
            "https://cdn.worldweatheronline.com/images/wsymbols01_png_64/wsymbol_0004_black_low_cloud.png"
        ],
        "weather_descriptions": [
            "Overcast "
        ],
        "astro": {
            "sunrise": "07:06 AM",
            "sunset": "06:17 PM",
            "moonrise": "11:24 AM",
            "moonset": "08:14 PM",
            "moon_phase": "Waxing Crescent",
            "moon_illumination": 11
        },
        "air_quality": {
            "co": "226",
            "no2": "13.5",
            "o3": "78",
            "so2": "1.3",
            "pm2_5": "9.5",
            "pm10": "9.7",
            "us-epa-index": "1",
            "gb-defra-index": "1"
        },
        "wind_speed": 16,
        "wind_degree": 309,
        "wind_dir": "NW",
        "pressure": 1011,
        "precip": 0,
        "humidity": 59,
        "cloudcover": 98,
        "feelslike": 26,
        "uv_index": 0,
        "visibility": 10,
        "is_day": "yes"
    }
}
```

In [ ]:
@tool 
def get_weather_data(city:str) -> str:
    """
    This function fetches current weather information of a city
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={weatherstack_api_key}&query={city}"
    )   

    response = requests.get(url)
    data = response.json()

    if 'current' not in data:
        return f"Could not fetch weather data for {city}"

    return (
    f"City: {city}\n"
    f"Temperature: {data['current']['temperature']}°C\n"
    f"Weather: {data['current']['weather_descriptions'][0]}\n"
    f"Humidity: {data['current']['humidity']}%"
    )
    


In [ ]:
weather = get_weather_data('New york city')
weather

## Step 5: Initialize the Gemini Language Model (LLM)

- ChatGoogleGenerativeAI: Connects LangChain to Google's Gemini models.
- model="gemini-3.8-flash": Selects the Gemini 2.5 Flash model.
- temperature=0: Makes responses more consistent.
- google_api_key=GOOGLE_API_KEY: Authenticates requests using your Gemini API key.

In [ ]:
llm = ChatGoogleGenerativeAI(
    model='gemini-3.8-flash',
    temperature=0,
    google_api_key=gemini_api_key
)

## 6. Load the Agent Prompt

- `hub`: Provides access to prompts shared on LangChain Hub.
- `hub.pull("hwchase17/react")`: Retrieves a pre-built ReAct prompt for the agent.
- **ReAct (Reasoning and Acting):** Allows the agent to choose tools, observe their results, and use them to complete a task.

In [ ]:
prompt = hub.pull("hwchase17/react")


## 7. Define the Tools List

The tools list contains the tools available to the AI agent.

- `tools`: Stores the available tools in a Python list.
- `search_tool`: Allows the agent to search the internet using Tavily.
- `get_weather_data`: Allows the agent to retrieve current weather information using Weatherstack.

In [ ]:
tools = [
    tavily_tool, 
    get_weather_data
]

## 8. Create the AI Agent

- `create_react_agent`: Creates an agent that follows the ReAct approach.
- `llm=llm`: Sets Gemini as the language model.
- `tools=tools`: Gives the agent access to Tavily Search and the Weatherstack tool.
- `prompt=prompt`: Provides the instructions the agent follows when using its tools.

In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

## 9. Create the Agent Executor

- `AgentExecutor`: Runs the agent and manages its tool calls.
- `agent=agent`: Specifies the agent to execute.
- `tools=tools`: Provides the available tools.
- `verbose=True`: Displays the agent's intermediate steps and tool usage.

In [ ]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
        handle_parsing_errors=True

)

In [ ]:
response = agent_executor.invoke({
    "input": (
        "Find the latest news on AI"
        "Also find how current global wars are affecting the data centers"
    )
})

print("\n========================")
print("FINAL OUTPUT")
print("========================\n")

print(response['output'])